# Errors

> fastllm errors.

In [ ]:
#| default_exp errors

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Optional
import json
import httpx

In [ ]:
#| hide
from fastcore.test import *

## Purpose And Design

`errors.py` defines the error contract used across the entire library.

### Why this module exists

Providers return different error shapes for HTTP and streaming failures. If each layer handled raw provider payloads directly, calling code would need provider-specific branches everywhere.

### Architectural fit

- Upstream inputs: raw `httpx` exceptions and SSE error events.
- Downstream consumers: `transport`, dynamic ops in `oapi`, provider `clients`, and high-level entrypoints.

### Key design choices

- `APIError` carries normalized fields (`status_code`, `error_type`, `code`, `request_id`, `retryable`, `raw`).
- Best-effort parsing keeps useful diagnostics without overfitting to one vendor.
- Context enrichment (`provider`, `model`, `endpoint`) helps debugging in multi-provider runs.

### Reader outcome

After this notebook, you should be able to trace where any failure is normalized and how error metadata remains consistent from low-level transport to user-facing calls.

### `FastLLMError`

Base fastllm error.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class FastLLMError(Exception):
    "Base fastllm error."

### `UnsupportedCapabilityError`

Raised when a requested feature is unsupported.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `clients`, `files`, `oapi`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class UnsupportedCapabilityError(FastLLMError):
    "Raised when a requested feature is unsupported."

### `ProtocolError`

Raised when provider payloads do not match expected protocol shape.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `normalize`, `transport`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class ProtocolError(FastLLMError):
    "Raised when provider payloads do not match expected protocol shape."

### `SpecError`

Raised when an OpenAPI spec cannot be parsed as expected.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Encapsulates one coherent concept to reduce repeated shape logic elsewhere.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `spec`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
class SpecError(FastLLMError):
    "Raised when an OpenAPI spec cannot be parsed as expected."

### `_to_text`

Best-effort stringify helper.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_to_text(v: Any)`
- Returns: `str`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _to_text(v: Any) -> str:
    "Best-effort stringify helper."
    if v is None: return ""
    if isinstance(v, str): return v
    try: return json.dumps(v, ensure_ascii=False)
    except Exception: return str(v)

### `_retryable`

Classify transient/retryable API failures.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_retryable(status_code: Optional[int], error_type: Any, code: Any, message: str)`
- Returns: `bool`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _retryable(status_code: Optional[int], error_type: Any, code: Any, message: str) -> bool:
    "Classify transient/retryable API failures."
    if isinstance(status_code, int) and (status_code >= 500 or status_code in (408, 409, 425, 429)): return True
    t = str(error_type or "").lower()
    c = str(code or "").lower()
    m = (message or "").lower()
    hints = ("server_error", "internal", "overload", "rate_limit", "timeout", "unavailable", "temporar")
    if any(h in t for h in hints): return True
    if any(h in c for h in hints): return True
    if any(h in m for h in ("try again", "server error", "temporar", "timeout", "overloaded", "unavailable")): return True
    return False

### `_req_id`

Extract provider request id header when available.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_req_id(headers: Optional[httpx.Headers])`
- Returns: `str`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _req_id(headers: Optional[httpx.Headers]) -> str:
    "Extract provider request id header when available."
    if headers is None: return ""
    for k in ("x-request-id", "request-id", "anthropic-request-id", "x-goog-request-id"):
        v = headers.get(k)
        if v: return str(v)
    return ""

### `_parse_http_error_response`

Parse common HTTP API error shapes to (message, error_type, code, raw).

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_parse_http_error_response(resp: httpx.Response)`
- Returns: `tuple[str, str, Any, Any]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _parse_http_error_response(resp: httpx.Response) -> tuple[str, str, Any, Any]:
    "Parse common HTTP API error shapes to (message, error_type, code, raw)."
    raw: Any
    try:
        raw = resp.json()
    except Exception:
        raw = resp.text

    msg, et, code, status = "", "", None, None
    if isinstance(raw, dict):
        err = raw.get("error")
        if isinstance(err, dict):
            msg = str(err.get("message") or raw.get("message") or raw.get("detail") or "")
            status = err.get("status", raw.get("status"))
            et = str(err.get("type") or status or raw.get("type") or "")
            code = err.get("code", raw.get("code"))
        elif err is not None:
            msg = _to_text(err)
            status = raw.get("status")
            et = str(raw.get("type") or status or "")
            code = raw.get("code")
        else:
            msg = str(raw.get("message") or raw.get("detail") or raw.get("error_description") or "")
            status = raw.get("status")
            et = str(raw.get("type") or status or "")
            code = raw.get("code")
        if not msg: msg = _to_text(raw)
    else:
        msg = _to_text(raw)
    # Prefer semantic provider codes (e.g. Gemini `status`) over numeric HTTP-like codes.
    if (code is None or isinstance(code, int)) and isinstance(status, str) and status:
        code = status
    if code in (None, "") and isinstance(et, str) and et:
        code = et
    return msg, et, code, raw

### `_parse_sse_error_event`

Parse common SSE `error` event shapes to (message, error_type, code, raw).

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_parse_sse_error_event(event: Any)`
- Returns: `tuple[str, str, Any, Any]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def _parse_sse_error_event(event: Any) -> tuple[str, str, Any, Any]:
    "Parse common SSE `error` event shapes to (message, error_type, code, raw)."
    raw = event
    e = event.get("error") if isinstance(event, dict) and isinstance(event.get("error"), dict) else (
        event if isinstance(event, dict) else {"message": _to_text(event)})
    msg = str(e.get("message") or e.get("detail") or _to_text(e))
    et = str(e.get("type") or e.get("status") or "")
    code = e.get("code")
    if code in (None, "") and et: code = et
    return msg, et, code, raw

### `APIError`

Structured cross-provider API failure object.

**Why it exists**
- Callers need a single error shape across HTTP and stream failures from different providers.

**Design Notes**
- Includes normalized diagnostics (`status_code`, `code`, `error_type`, `request_id`, `retryable`) plus raw payload.
- Context fields (`provider`, `model`, `endpoint`) make failures actionable in multi-provider systems.

**Connections**
- Raised by transport/oapi/client layers and surfaced to high-level consumers.

In [ ]:
#| export
class APIError(FastLLMError):
    "Structured provider/API error with context."
    def __init__(self, message: str, *, provider: str = "", model: str = "", endpoint: str = "",
        status_code: Optional[int] = None, error_type: str = "", code: Any = None, request_id: str = "",
        retryable: Optional[bool] = None, raw: Any = None):
        self.message = message or "API request failed"
        self.provider = provider or ""
        self.model = model or ""
        self.endpoint = endpoint or ""
        self.status_code = status_code
        self.error_type = error_type or ""
        self.code = code
        self.request_id = request_id or ""
        self.retryable = _retryable(status_code, self.error_type, self.code, self.message) if retryable is None else bool(retryable)
        self.raw = raw
        super().__init__(self.__str__())

    def with_context(self, *, provider: str = "", model: str = "", endpoint: str = "") -> "APIError":
        "Return a copy with missing context fields filled."
        return APIError(
            self.message,
            provider=self.provider or provider,
            model=self.model or model,
            endpoint=self.endpoint or endpoint,
            status_code=self.status_code,
            error_type=self.error_type,
            code=self.code,
            request_id=self.request_id,
            retryable=self.retryable,
            raw=self.raw,
        )

    def __str__(self):
        bits = []
        if self.provider: bits.append(self.provider)
        if self.endpoint: bits.append(self.endpoint)
        if self.model: bits.append(f"model={self.model}")
        if self.status_code is not None: bits.append(f"status={self.status_code}")
        if self.error_type: bits.append(f"type={self.error_type}")
        if self.code not in (None, ""): bits.append(f"code={self.code}")
        if self.request_id: bits.append(f"request_id={self.request_id}")
        if self.retryable: bits.append("retryable=True")
        pref = " ".join(bits)
        return f"{pref}: {self.message}" if pref else self.message

### `api_error_from_http`

Converts `httpx.HTTPStatusError` into `APIError`.

**Why it exists**
- HTTP error payloads differ by provider and endpoint.

**Design Notes**
- Parses best-effort structured fields and computes retryability hints.

**Connections**
- Core adapter used by transport/oapi/client error paths.

In [ ]:
#| export
def api_error_from_http(exc: httpx.HTTPStatusError, *, provider: str = "", model: str = "", endpoint: str = "") -> APIError:
    "Build APIError from httpx HTTPStatusError."
    resp = exc.response
    msg, et, code, raw = _parse_http_error_response(resp)
    ep = endpoint
    if not ep and resp.request is not None:
        ep = f"{resp.request.method.upper()} {resp.request.url.path}"
    return APIError(
        msg,
        provider=provider,
        model=model,
        endpoint=ep,
        status_code=resp.status_code,
        error_type=et,
        code=code,
        request_id=_req_id(resp.headers),
        raw=raw,
    )

### `api_error_from_event`

Build APIError from provider SSE/event-level error payload.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `api_error_from_event(event: Any, *, provider: str, model: str, endpoint: str)`
- Returns: `APIError`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: `normalize`.
- Module downstream: `transport`, `spec`, `oapi`, `normalize`, `clients`, `highlevel`, `files`.

In [ ]:
#| export
def api_error_from_event(event: Any, *, provider: str = "", model: str = "", endpoint: str = "") -> APIError:
    "Build APIError from provider SSE/event-level error payload."
    msg, et, code, raw = _parse_sse_error_event(event)
    return APIError(msg, provider=provider, model=model, endpoint=endpoint, error_type=et, code=code, raw=raw)

## Quick Checks

In [ ]:
import fastllm.errors as m
for nm in ['FastLLMError', 'UnsupportedCapabilityError', 'ProtocolError', 'SpecError', 'APIError', 'api_error_from_http', 'api_error_from_event']: assert hasattr(m, nm), nm
from fastllm.errors import APIError
e = APIError('oops', provider='openai', model='gpt-5-mini', endpoint='responses.create', status_code=400)
assert e.status_code==400 and e.provider=='openai'